# Adım 5: Özellik Mühendisliği (Feature Engineering)

Gold Delta katmanına kaydedilen feature seti incelenir, en az 5 yeni anlamlı özellik üretilir ve Delta Lake'e kaydedilir.

| Özellik | Kaynak | Mantık |
|---------|--------|--------|
| `is_airport_trip` | zone adı | Havaalanı yolculuğu flag |
| `fare_per_mile` | fare / distance | Mil başına ücret verimliliği |
| `is_rush_hour` | pickup_hour | Sabah/akşam zirve saatleri |
| `is_weekend` | pickup_day_of_week | Hafta sonu flag |
| `distance_bin` | trip_distance | Mesafe kategorisi |
| `cross_borough` | PU/DO borough | Farklı ilçe seyahati |
| `passenger_group` | passenger_count | Yolcu grubu kategorisi |

## 1. Kurulum & Import

In [ ]:
import subprocess, sys
for pkg in ["pandas", "numpy", "pyarrow==15.0.2", "scikit-learn", "matplotlib", "seaborn"]:
    r = subprocess.run([sys.executable, "-m", "pip", "install", pkg, "-q"],
                       capture_output=True, text=True)
    print(f"{'✅' if r.returncode==0 else '❌'} {pkg}")

In [ ]:
import os, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

warnings.filterwarnings("ignore")
plt.style.use("seaborn-v0_8-darkgrid")
plt.rcParams.update({"figure.dpi": 110, "font.size": 11})

PROJECT_ROOT = Path(os.getcwd()).parent
SILVER_PATH  = PROJECT_ROOT / "data/delta/silver/yellow_taxi_trips"
GOLD_PATH    = PROJECT_ROOT / "data/delta/gold/fare_features"
FEATURES_PATH = PROJECT_ROOT / "data/delta/gold/engineered_features"
PARQUET_PATH = PROJECT_ROOT / "data/raw/yellow_tripdata_2023-01.parquet"

print(f"Proje kök: {PROJECT_ROOT}")

## 2. Veri Yükleme

In [ ]:
import pyarrow.parquet as pq

def load_data(limit=50_000):
    # Gold Delta dene
    if list(GOLD_PATH.rglob("*.parquet")):
        df = pd.read_parquet(GOLD_PATH, engine="pyarrow")
        src = "Gold Delta"
    # Silver Delta dene
    elif list(SILVER_PATH.rglob("*.parquet")):
        df = pd.read_parquet(SILVER_PATH, engine="pyarrow")
        src = "Silver Delta"
    # Ham Parquet dene
    elif PARQUET_PATH.exists():
        raw = pq.read_table(PARQUET_PATH).to_pandas()
        raw["pickup_datetime"]    = pd.to_datetime(raw["tpep_pickup_datetime"], errors="coerce")
        raw["dropoff_datetime"]   = pd.to_datetime(raw["tpep_dropoff_datetime"], errors="coerce")
        raw["trip_duration_minutes"] = (raw["dropoff_datetime"] - raw["pickup_datetime"]).dt.total_seconds() / 60
        raw["pickup_hour"]        = raw["pickup_datetime"].dt.hour
        raw["pickup_day_of_week"] = raw["pickup_datetime"].dt.dayofweek + 1
        raw["pickup_borough"]     = "Unknown"
        raw["dropoff_borough"]    = "Unknown"
        raw["pickup_zone"]        = "Unknown"
        raw["dropoff_zone"]       = "Unknown"
        df = raw[
            (raw["trip_distance"] > 0) & (raw["trip_distance"] <= 100) &
            (raw["fare_amount"]   > 0) & (raw["fare_amount"]   <= 500) &
            (raw["trip_duration_minutes"] > 0) & (raw["trip_duration_minutes"] <= 240)
        ].copy()
        df = df.rename(columns={"fare_amount": "label"})
        src = "Ham Parquet"
    else:
        raise FileNotFoundError("Veri kaynağı bulunamadı!")

    if len(df) > limit:
        df = df.sample(limit, random_state=42).reset_index(drop=True)
    return df, src

df, kaynak = load_data()
print(f"Kaynak : {kaynak}")
print(f"Satır  : {len(df):,}  |  Sütun: {len(df.columns)}")
print(f"Sütunlar: {list(df.columns)}")

## 3. Mevcut Özellikler (Gold'dan Gelenler)

In [ ]:
gold_features = [
    "label", "trip_distance", "passenger_count", "pickup_hour",
    "pickup_day_of_week", "PULocationID", "DOLocationID",
    "RatecodeID", "payment_type", "pickup_borough", "dropoff_borough",
    "pickup_zone", "dropoff_zone", "is_airport_trip"
]
mevcut = [c for c in gold_features if c in df.columns]
eksik  = [c for c in gold_features if c not in df.columns]

print("Mevcut özellikler:")
for c in mevcut:
    print(f"  ✅ {c}")
if eksik:
    print("\nEksik (kaynak bağımlı):")
    for c in eksik:
        print(f"  ⚠️  {c}")

## 4. Yeni Özellik Üretimi

### Özellik 1: `fare_per_mile` — Mil Başına Ücret
*Mantık: Kısa ama pahalı yolculuklar (havaalanı, tünel) ile uzun ama ucuz yolculukları ayırt eder. Ücret tahmini için güçlü bir normalize değişken.*

In [ ]:
label_col = "label" if "label" in df.columns else "fare_amount"

df["fare_per_mile"] = (df[label_col] / df["trip_distance"].clip(lower=0.1)).round(2)
# Aykırı değerleri kırp
df["fare_per_mile"] = df["fare_per_mile"].clip(upper=50)

print("fare_per_mile — Özet:")
print(df["fare_per_mile"].describe().round(2).to_string())

fig, ax = plt.subplots(figsize=(8, 4))
df["fare_per_mile"].clip(upper=20).hist(bins=60, color="#3498db", alpha=0.8, ax=ax)
ax.set_title("Dağılım: fare_per_mile ($/mil)")
ax.set_xlabel("$/mil")
ax.set_ylabel("Frekans")
plt.tight_layout()
plt.show()

### Özellik 2: `is_rush_hour` — Zirve Saat Flag'i
*Mantık: Sabah (7-9) ve akşam (16-19) zirve saatlerinde trafik yoğunluğu artар, seyahat süresi uzar ve fare_amount yükselebilir.*

In [ ]:
if "pickup_hour" in df.columns:
    df["is_rush_hour"] = df["pickup_hour"].apply(
        lambda h: 1 if (7 <= h <= 9) or (16 <= h <= 19) else 0
    )
    oran = df["is_rush_hour"].mean() * 100
    print(f"Zirve saat oranı : {oran:.1f}%")
    print(df.groupby("is_rush_hour")[label_col].agg(["mean","count"]).round(2))

### Özellik 3: `is_weekend` — Hafta Sonu Flag'i
*Mantık: Hafta sonu (Cumartesi=7, Pazar=1) yolculukları mesafe ve amaç açısından iş günlerinden farklıdır.*

In [ ]:
if "pickup_day_of_week" in df.columns:
    df["is_weekend"] = df["pickup_day_of_week"].apply(
        lambda d: 1 if d in [1, 7] else 0
    )
    print("Hafta içi vs Hafta sonu ortalama ücret:")
    print(df.groupby("is_weekend")[label_col].agg(["mean","median","count"]).round(2))

### Özellik 4: `distance_bin` — Mesafe Kategorisi
*Mantık: Mesafeyi kategorik gruplara ayırmak, doğrusal olmayan ilişkileri yakalamak için modele yardımcı olur.*

In [ ]:
bins   = [0, 1, 3, 7, 15, 100]
labels = ["cok_kisa", "kisa", "orta", "uzun", "cok_uzun"]

df["distance_bin"] = pd.cut(df["trip_distance"], bins=bins,
                             labels=labels, right=True)

print("Mesafe Kategorileri:")
tablo = df.groupby("distance_bin", observed=True)[label_col].agg(["count","mean"]).round(2)
tablo["oran%"] = (tablo["count"] / len(df) * 100).round(1)
print(tablo.to_string())

### Özellik 5: `cross_borough` — Farklı İlçe Yolculuğu
*Mantık: Biniş ve iniş farklı ilçelerdeyse genellikle daha uzun ve pahalı yolculuklardır.*

In [ ]:
if "pickup_borough" in df.columns and "dropoff_borough" in df.columns:
    df["cross_borough"] = (
        (df["pickup_borough"] != df["dropoff_borough"]) &
        (df["pickup_borough"] != "Unknown") &
        (df["dropoff_borough"] != "Unknown")
    ).astype(int)
    print("cross_borough — Ortalama Ücret Karşılaştırması:")
    print(df.groupby("cross_borough")[label_col].agg(["mean","count"]).round(2))
else:
    df["cross_borough"] = 0
    print("Borough bilgisi mevcut değil, cross_borough=0 atandı.")

### Özellik 6: `passenger_group` — Yolcu Grubu
*Mantık: Yolcu sayısını gruplara ayırmak, bireysel vs grup seyahatlerini ayırt eder.*

In [ ]:
def passenger_group(n):
    if pd.isna(n) or n <= 1: return "tek"
    if n <= 3: return "kucuk_grup"
    return "buyuk_grup"

if "passenger_count" in df.columns:
    df["passenger_group"] = df["passenger_count"].apply(passenger_group)
    print("passenger_group dağılımı:")
    print(df["passenger_group"].value_counts().to_string())

### Özellik 7: `is_airport_trip` (Gold'da yoksa üret)
*Mantık: Havaalanı yolculukları sabit ücret tarife içerebilir.*

In [ ]:
if "is_airport_trip" not in df.columns:
    if "pickup_zone" in df.columns and "dropoff_zone" in df.columns:
        df["is_airport_trip"] = (
            df["pickup_zone"].str.lower().str.contains("airport", na=False) |
            df["dropoff_zone"].str.lower().str.contains("airport", na=False)
        ).astype(int)
    else:
        df["is_airport_trip"] = 0

print(f"is_airport_trip=1 oranı: {df['is_airport_trip'].mean()*100:.1f}%")
print(df.groupby("is_airport_trip")[label_col].agg(["mean","count"]).round(2))

## 5. Üretilen Özelliklerin Özeti

In [ ]:
yeni_ozellikler = [
    ("fare_per_mile",    "Sayısal", "Mil başına ücret — normalize verimliliği"),
    ("is_rush_hour",     "İkili",   "Zirve saat flag (07-09 / 16-19)"),
    ("is_weekend",       "İkili",   "Hafta sonu flag (Cts=7, Paz=1)"),
    ("distance_bin",     "Kategorik","Mesafe grubu (çok kısa → çok uzun)"),
    ("cross_borough",    "İkili",   "Farklı ilçe yolculuğu flag"),
    ("passenger_group",  "Kategorik","Yolcu grubu (tek/küçük/büyük)"),
    ("is_airport_trip",  "İkili",   "Havaalanı yolculuğu flag"),
]

print(f"{'Özellik':<20} {'Tip':<12} {'Mantık'}")
print("-" * 65)
for ad, tip, mantik in yeni_ozellikler:
    mevcut_flag = "✅" if ad in df.columns else "⚠️ "
    print(f"  {mevcut_flag} {ad:<18} {tip:<12} {mantik}")

print(f"\nDataFrame boyutu: {df.shape}")

## 6. Feature Korelasyon Analizi

In [ ]:
from sklearn.preprocessing import LabelEncoder

analiz_df = df.copy()

# Kategorik özellikleri encode et
for col in ["distance_bin", "passenger_group"]:
    if col in analiz_df.columns:
        analiz_df[col] = LabelEncoder().fit_transform(analiz_df[col].astype(str))

sayisal_cols = [
    label_col, "fare_per_mile", "is_rush_hour", "is_weekend",
    "distance_bin", "cross_borough", "is_airport_trip", "trip_distance"
]
mevcut_cols = [c for c in sayisal_cols if c in analiz_df.columns]

corr = analiz_df[mevcut_cols].corr()[[label_col]].drop(label_col)
corr = corr.sort_values(label_col, ascending=True)

fig, ax = plt.subplots(figsize=(8, 5))
colors = ["#e74c3c" if v < 0 else "#2ecc71" for v in corr[label_col]]
corr[label_col].plot.barh(ax=ax, color=colors, alpha=0.85)
ax.axvline(0, color="black", linewidth=0.8)
ax.set_title(f"{label_col} ile Korelasyon")
ax.set_xlabel("Pearson Korelasyonu")
for i, (idx, row) in enumerate(corr.iterrows()):
    ax.text(row[label_col] + 0.005, i, f"{row[label_col]:.3f}", va="center", fontsize=9)
plt.tight_layout()
plt.show()

## 7. Delta Lake'e Kaydetme

In [ ]:
# Kategorik sütunları string'e çevir (parquet uyumluluğu)
kayit_df = df.copy()
for col in ["distance_bin", "passenger_group"]:
    if col in kayit_df.columns:
        kayit_df[col] = kayit_df[col].astype(str)

FEATURES_PATH.mkdir(parents=True, exist_ok=True)
out_file = FEATURES_PATH / "engineered_features.parquet"

kayit_df.to_parquet(out_file, index=False, engine="pyarrow")

print(f"✅ Kayıt tamamlandı!")
print(f"   Konum  : {out_file}")
print(f"   Satır  : {len(kayit_df):,}")
print(f"   Sütun  : {len(kayit_df.columns)}")
print(f"   Boyut  : {out_file.stat().st_size / 1024:.1f} KB")

## 8. Doğrulama — Kaydedilen Dosyayı Oku

In [ ]:
dogrulama = pd.read_parquet(out_file, engine="pyarrow")
print(f"✅ Doğrulama: {len(dogrulama):,} satır, {len(dogrulama.columns)} sütun")
print("\nİlk 3 satır:")
print(dogrulama.head(3).to_string())

## 9. Özet & Teslim Edilecekler

| # | Özellik | Tip | Teslim |
|---|---------|-----|--------|
| 1 | `fare_per_mile` | Sayısal | ✅ |
| 2 | `is_rush_hour` | İkili | ✅ |
| 3 | `is_weekend` | İkili | ✅ |
| 4 | `distance_bin` | Kategorik | ✅ |
| 5 | `cross_borough` | İkili | ✅ |
| 6 | `passenger_group` | Kategorik | ✅ |
| 7 | `is_airport_trip` | İkili | ✅ |

**Delta Lake kayıt yolu:** `data/delta/gold/engineered_features/`

---
> **Sonraki adım →** Adım 6: Makine Öğrenmesi — Çoklu Model Karşılaştırma